# Evaluate the fine-tuned model (epoch 300) with the reference sampler

Generates 10,000 images with the same 999-step sampler as the reference notebook and computes FID@2k (5 sets), FID at 2k-10k, FID-infinity, KID, Inception Score, precision/recall and a nearest-neighbour check.

In [1]:
!pip install -q torch-fidelity diffusers

In [2]:
RUN       = "G_finetune"
WEIGHTS   = "ema.pt"
ARCH      = "pretrained"
SCHEDULE  = "linear"
N_TOTAL   = 10000
SEED      = 0
BS        = 250

In [3]:
import os, sys, glob, json, math, time, threading
from concurrent.futures import ThreadPoolExecutor
import numpy as np, torch, torchvision, kagglehub
import torchvision.transforms as T
from torchvision.utils import save_image
from tqdm.auto import tqdm
from diffusers import UNet2DModel
for m in [k for k in list(sys.modules) if k.startswith("torchmetrics")]:
    del sys.modules[m]
import torch_fidelity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.inception import InceptionScore

assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2"
DEVS = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
DEV = DEVS[0]
OUT = f"/kaggle/working/eval_{RUN}_{WEIGHTS.replace('.pt', '')}"; os.makedirs(OUT, exist_ok=True)
print("GPUs:", [torch.cuda.get_device_name(d) for d in DEVS])

wp = sorted(glob.glob(f"/kaggle/input/**/{RUN}/{WEIGHTS}", recursive=True))
assert wp, f"Could not find {RUN}/{WEIGHTS} under /kaggle/input - attach the training notebook (Add Input -> Notebooks)."
print("weights:", wp[0])

hits = sorted(glob.glob("/kaggle/input/**/flowers-recognition", recursive=True))
path = hits[0] if hits else kagglehub.dataset_download("alxmamaev/flowers-recognition")
tf = T.Compose([T.ToTensor(), T.Resize((32, 32), interpolation=T.InterpolationMode.BICUBIC, antialias=True)])
ds = torchvision.datasets.ImageFolder(root=path, transform=tf)
real_u8 = (torch.stack([ds[i][0] for i in range(len(ds))]).clamp(0, 1) * 255).round().to(torch.uint8)
print("real:", tuple(real_u8.shape))

TSTEPS = 1000
def linear_betas(T_):
    s = 1000 / T_
    return torch.linspace(s * 1e-4, s * 0.02, T_, dtype=torch.float32)
def cosine_betas(T_, s=0.008):
    x = torch.arange(T_ + 1, dtype=torch.float64) / T_
    ab = torch.cos((x + s) / (1 + s) * math.pi / 2) ** 2
    ab = ab / ab[0]
    return (1 - ab[1:] / ab[:-1]).clamp(max=0.999).float()
beta = (cosine_betas if SCHEDULE == "cosine" else linear_betas)(TSTEPS)
alpha = 1 - beta
a_bar = torch.cumprod(alpha, 0)
sqrt_1m_ab = (1 - a_bar).sqrt()
one_by_sqrt_alpha = 1 / alpha.sqrt()

def build_model(arch):
    if arch == "pretrained":
        return UNet2DModel.from_pretrained("google/ddpm-cifar10-32")
    if arch == "orig":
        down = ("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D")
        up   = ("AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D")
    elif arch == "attn16":
        down = ("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D")
        up   = ("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D")
    return UNet2DModel(in_channels=3, out_channels=3, block_out_channels=[64, 128, 256, 256],
                       layers_per_block=2, down_block_types=down, up_block_types=up)
sdict = torch.load(wp[0], map_location="cpu")
nets = []
for d in DEVS:
    m = build_model(ARCH); m.load_state_dict(sdict); nets.append(m.to(d).eval())
print(f"params: {sum(p.numel() for p in nets[0].parameters())/1e6:.2f}M")

GPUs: ['Tesla T4', 'Tesla T4']
weights: /kaggle/input/notebooks/malharinamdar/ddpm-task/G_finetune/ema.pt
real: (4317, 3, 32, 32)


params: 35.75M


In [4]:
@torch.no_grad()
def sample_on(i, n, seed, bs, tag):
    d, m = DEVS[i], nets[i]
    b_, s1m, oba = beta.to(d), sqrt_1m_ab.to(d), one_by_sqrt_alpha.to(d)
    g = torch.Generator(device=d).manual_seed(seed * 1000 + i); outs = []
    for s in range(0, n, bs):
        b = min(bs, n - s)
        x = torch.randn((b, 3, 32, 32), device=d, generator=g)
        for t in tqdm(range(TSTEPS - 1, 0, -1), desc=f"gpu{i} batch {s//bs+1}/{(n+bs-1)//bs}", position=i, leave=False):
            ts = torch.full((b,), t, dtype=torch.long, device=d)
            z = torch.randn(x.shape, device=d, generator=g) if t > 1 else torch.zeros_like(x)
            eps = m(x, ts).sample
            x = oba[t] * (x - (b_[t] / s1m[t]) * eps) + b_[t].sqrt() * z
        x = torch.nan_to_num(x, nan=0.0).clamp(-1, 1)
        outs.append(((x + 1) / 2 * 255).round().to(torch.uint8).cpu())
        np.save(f"{OUT}/{tag}_gpu{i}.npy", torch.cat(outs).numpy())
    return torch.cat(outs)

def sample(n, seed, bs, tag="samples"):
    k = len(DEVS); shares = [n // k + (1 if i < n % k else 0) for i in range(k)]
    with ThreadPoolExecutor(k) as ex:
        parts = list(ex.map(lambda i: sample_on(i, shares[i], seed, bs, tag), range(k)))
    return torch.cat(parts)

t0 = time.time()
gen = sample(N_TOTAL, SEED, BS)
sample_minutes = (time.time() - t0) / 60
np.save(f"{OUT}/samples_all.npy", gen.numpy())
save_image(gen[:64].float() / 255, f"{OUT}/samples_grid.png", nrow=8)
print(f"sampled {tuple(gen.shape)} in {sample_minutes:.1f} min")

sampled (10000, 3, 32, 32) in 237.5 min


In [7]:
def fid_kid(a_u8, b_u8):
    fid = FrechetInceptionDistance(feature=2048).to(DEV)
    kid = KernelInceptionDistance(subset_size=min(1000, len(a_u8), len(b_u8))).to(DEV)
    for x, flag in ((a_u8, True), (b_u8, False)):
        for i in range(0, len(x), 500):
            xb = x[i:i + 500].to(DEV); fid.update(xb, real=flag); kid.update(xb, real=flag)
    km, ks = kid.compute()
    return fid.compute().item(), km.item(), ks.item()

res = dict(run=RUN, weights=WEIGHTS, arch=ARCH, schedule=SCHEDULE, n_total=len(gen),
           sample_minutes=round(sample_minutes, 1), gpus=len(DEVS))

r2k = [fid_kid(real_u8, gen[j:j + 2000]) for j in range(0, len(gen) - 1999, 2000)]
res["fid_2k"] = [round(r[0], 3) for r in r2k]; res["kid_2k"] = [r[1] for r in r2k]
res["fid_2k_mean"], res["fid_2k_std"] = float(np.mean(res["fid_2k"])), float(np.std(res["fid_2k"]))
res["kid_2k_mean"], res["kid_2k_std"] = float(np.mean(res["kid_2k"])), float(np.std(res["kid_2k"]))
print("FID@2k:", res["fid_2k"], " KID@2k mean:", res["kid_2k_mean"])

Ns = [n for n in (2000, 4000, 6000, 8000, 10000) if n <= len(gen)]
fid_at = {}
for n in Ns:
    vals = [fid_kid(real_u8, gen[torch.randperm(len(gen), generator=torch.Generator().manual_seed(r))[:n]])[0]
            for r in range(3 if n < len(gen) else 1)]
    fid_at[n] = float(np.mean(vals)); print(f"FID@{n}: {fid_at[n]:.3f}")
res["fid_at_N"] = fid_at
if len(Ns) >= 2:
    b, a = np.polyfit([1 / n for n in Ns], [fid_at[n] for n in Ns], 1)
    res["fid_inf"], res["fid_inf_slope"] = float(a), float(b)
    print("FID_inf (extrapolated):", round(a, 3))

perm = torch.randperm(len(real_u8), generator=torch.Generator().manual_seed(0)); h = len(real_u8) // 2
res["floor_fid_2k"], res["floor_kid_2k"], _ = fid_kid(real_u8[perm[:h]], real_u8[perm[h:2 * h]])

json.dump(res, open(f"{OUT}/results.json", "w"), indent=2)
print("core results saved")

def inception_score(x_u8):
    m = InceptionScore().to(DEV)
    for i in range(0, len(x_u8), 500): m.update(x_u8[i:i + 500].to(DEV))
    mu, sd = m.compute(); return mu.item(), sd.item()
try:
    res["is_mean"], res["is_std"] = inception_score(gen)
    res["is_real_mean"], res["is_real_std"] = inception_score(real_u8)
except Exception as e:
    res["is_error"] = repr(e); print("IS failed:", e)

class U8(torch.utils.data.Dataset):
    def __init__(self, x): self.x = x
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i]
try:
    prc = torch_fidelity.calculate_metrics(input1=U8(gen[:2000]), input2=U8(real_u8), prc=True,
                                           cuda=True, batch_size=250, verbose=False)
    res["precision"], res["recall"] = prc["precision"], prc["recall"]
except Exception as e:
    res["prc_error"] = repr(e); print("precision/recall failed:", e)

R = real_u8.float().flatten(1) / 255; G = gen[:64].float().flatten(1) / 255
exact = "donot_use_mm_for_euclid_dist"
d_gen, idx = torch.cdist(G, R, compute_mode=exact).min(1)
q = torch.randperm(len(R), generator=torch.Generator().manual_seed(0))[:64]
dr = torch.cdist(R[q], R, compute_mode=exact); dr[torch.arange(64), q] = float("inf")
res["nn_dist_gen"], res["nn_dist_real"] = d_gen.mean().item(), dr.min(1).values.mean().item()
pairs = torch.stack([gen[:64], real_u8[idx]], 1).flatten(0, 1).float() / 255
save_image(pairs[:32], f"{OUT}/nearest_neighbours.png", nrow=8)

json.dump(res, open(f"{OUT}/results.json", "w"), indent=2)
print(json.dumps(res, indent=2))

FID@2k: [36.934, 34.427, 34.912, 34.29, 35.264]  KID@2k mean: 0.013596641272306443
FID@2000: 35.971
FID@4000: 29.395
FID@6000: 27.109
FID@8000: 26.135
FID@10000: 25.680
FID_inf (extrapolated): 22.91
core results saved


{
  "run": "G_finetune",
  "weights": "ema.pt",
  "arch": "pretrained",
  "schedule": "linear",
  "n_total": 10000,
  "sample_minutes": 237.5,
  "gpus": 2,
  "fid_2k": [
    36.934,
    34.427,
    34.912,
    34.29,
    35.264
  ],
  "kid_2k": [
    0.015278932638466358,
    0.012704884633421898,
    0.013630654662847519,
    0.012586886994540691,
    0.013781847432255745
  ],
  "fid_2k_mean": 35.1654,
  "fid_2k_std": 0.9500988580142589,
  "kid_2k_mean": 0.013596641272306443,
  "kid_2k_std": 0.0009675118684064146,
  "fid_at_N": {
    "2000": 35.97071075439453,
    "4000": 29.395275751749676,
    "6000": 27.109162012736004,
    "8000": 26.13513437906901,
    "10000": 25.67959213256836
  },
  "fid_inf": 22.909909089909856,
  "fid_inf_slope": 26049.92372055614,
  "floor_fid_2k": 22.61501693725586,
  "floor_kid_2k": -6.58464414300397e-05,
  "is_mean": 5.278594017028809,
  "is_std": 0.1790829449892044,
  "is_real_mean": 6.280346870422363,
  "is_real_std": 0.38180240988731384,
  "precision"